# 2-4. LCEL로 풀 RAG 파이프라인 (feat. 전자금융 표준약관)

⏱ **소요시간**: 30분

## 학습 목표
- LCEL(LangChain Expression Language) 파이프 `|` 로 full RAG 체인을 조립한다.
- `retriever | prompt | llm | parser` 구조를 이해하고 citation(근거 인용)을 포함하는 프롬프트를 설계한다.
- 전자금융 표준약관에 대해 5개 질의를 실행하고 결과를 **S3에서 재사용할 변수**에 저장한다.

## 핵심 키워드
`LCEL` `RunnablePassthrough` `RunnableParallel` `citation` `StrOutputParser` `capstone baseline`

> 🔒 본 노트북은 **캡스톤 베이스라인**을 구축한다. Day2 Session 3에서 이 마지막 셀의 변수를 이어서 사용한다.

In [ ]:
import sys; sys.path.insert(0, '../..')
from common import get_llm, get_embeddings, provider_badge
print(provider_badge())

## 1. 코퍼스 구성 — 전자금융 표준약관 발췌

실제 현장에서는 금융결제원/여신금융협회 등 공식 약관 PDF를 넣지만, 수업에서는 재배포가 어려우므로 **자체 축약 문서**로 대체한다.

In [ ]:
from langchain_core.documents import Document

E_FINANCE_TERMS = [
    Document(page_content='제1조(목적) 본 약관은 전자금융거래에 관한 사항을 정함으로써 이용자의 권익 보호와 거래의 안전을 확보함을 목적으로 한다.', metadata={'source': 'efinance_terms', 'article': '제1조'}),
    Document(page_content='제2조(정의) 전자금융거래란 금융회사 또는 전자금융업자가 전자적 장치를 통해 제공하는 금융상품 및 서비스를 자동화된 방식으로 이용하는 거래를 말한다.', metadata={'source': 'efinance_terms', 'article': '제2조'}),
    Document(page_content='제5조(청약철회) ① 이용자는 계약 체결 후 14일 이내에 서면 또는 전자문서로 청약을 철회할 수 있다. ② 다만 이미 안내된 서비스가 제공되었거나 영업일 기준 3일 이내 증정이 끝난 경우 철회가 제한될 수 있다.', metadata={'source': 'efinance_terms', 'article': '제5조'}),
    Document(page_content='제8조(수수료) 전자금융거래 이용 수수료는 거래 유형별로 사전에 안내되며, 수수료 인상 시 적용일 30일 전에 공지해야 한다.', metadata={'source': 'efinance_terms', 'article': '제8조'}),
    Document(page_content='제10조(분실신고) ① 이용자는 접근매체의 분실·도난 시 즉시 고객센터 또는 가장 가까운 영업점에 신고하여야 한다. ② 신고 접수 이전에 발생한 사고에 대해서도 선의무과실이 없는 한 금융회사가 보상책임을 부담한다.', metadata={'source': 'efinance_terms', 'article': '제10조'}),
    Document(page_content='제14조(이의신청) ① 이용자는 거래 내역에 이의가 있을 때 거래일로부터 1년 이내에 이의를 신청할 수 있다. ② 금융회사는 이의신청 접수일로부터 15영업일 이내에 결과를 통지한다.', metadata={'source': 'efinance_terms', 'article': '제14조'}),
    Document(page_content='제15조(개인정보보호) 금융회사는 개인정보보호법에 따라 이용자의 개인정보를 안전하게 관리해야 하며, 제3자 제공 시 사전 동의를 받아야 한다.', metadata={'source': 'efinance_terms', 'article': '제15조'}),
    Document(page_content='제17조(서비스 이용시간) 전자금융거래 서비스는 24시간 이용을 원칙으로 하되, 시스템 점검·장애·운영상 필요에 따라 일시적으로 제한될 수 있으며 사전 공지한다.', metadata={'source': 'efinance_terms', 'article': '제17조'}),
]
for i, d in enumerate(E_FINANCE_TERMS):
    d.metadata['chunk_id'] = i
print(f'약관 총 {len(E_FINANCE_TERMS)}개 조항 준비 완료')

In [ ]:
import shutil
from pathlib import Path
from langchain_community.vectorstores import Chroma

CHROMA_DIR = Path('./_store/efinance_rag')
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
CHROMA_DIR.parent.mkdir(parents=True, exist_ok=True)

vs = Chroma.from_documents(E_FINANCE_TERMS, embedding=get_embeddings(), persist_directory=str(CHROMA_DIR), collection_name='efinance_terms')
retriever = vs.as_retriever(search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 8, 'lambda_mult': 0.5})
print('인덱싱 완료:', vs._collection.count(), '문서')

## 2. citation 포함 프롬프트 + LCEL 체인

**정답 명칭이 `근거`를 가지지 않으면 `모르겠습니다` 로 답하도록 규정** — 금융권 컴플라이언스 기본 원칙.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

SYSTEM = '''당신은 금융회사의 내부 규정 담당자를 돕는 AI 어시스턴트입니다.
반드시 아래 [컨텍스트]에 근거해 답하세요.
규칙:
1. [컨텍스트]에 근거가 없으면 "제공된 약관에서 기술되지 않아 답변할 수 없습니다." 라고 답합니다.
2. 답변 마지막에 반드시 "[근거: 제X조]" 형식으로 출처를 모두 목록합니다.
3. 답변은 섹션/문장 단위로 간결하게 한국어로 작성합니다.'''

USER = '''[질문]
{question}

[컨텍스트]
{context}'''

prompt = ChatPromptTemplate.from_messages([('system', SYSTEM), ('user', USER)])

def format_context(docs):
    lines = []
    for d in docs:
        art = d.metadata.get('article', '알 수 없음')
        lines.append(f'[{art}] {d.page_content}')
    return '\n\n'.join(lines)

llm = get_llm(temperature=0)

# LCEL 체인 — 검색 결과를 들고 있으면 citation UI에 사용 가능
rag_chain = (
    RunnableParallel({
        'question': RunnablePassthrough(),
        'docs': retriever,
    })
    .assign(context=lambda x: format_context(x['docs']))
    .assign(answer=prompt | llm | StrOutputParser())
)
print('체인 구성 완료')

In [ ]:
# 확인 실행 — 1개 질문
out = rag_chain.invoke('청약철회 기간은 며칠이고, 제한되는 경우는 언제인가요?')
print('— Retrieved —')
for d in out['docs']:
    print(f'  [{d.metadata["article"]}] {d.page_content[:60]}…')
print('\n— Answer —')
print(out['answer'])

## 3. 5개 질문 일괄 실행 — 캡스톤 베이스라인

In [ ]:
CAPSTONE_QUESTIONS = [
    '청약철회 기간은 며칠인가?',
    '카드 등 접근매체 분실 시 신고 절차는 어떻게 되나?',
    '수수료를 인상하려면 며칠 전에 공지해야 하는가?',
    '거래 내역에 이의가 있을 때 언제까지 이의신청을 할 수 있고, 금융회사의 회신 기한은 얼마인가?',
    '서비스 이용시간은 어떻게 되며, 일시 제한 사유에는 무엇이 있는가?',
]

capstone_baseline = []
for q in CAPSTONE_QUESTIONS:
    result = rag_chain.invoke(q)
    capstone_baseline.append({
        'question': q,
        'answer': result['answer'],
        'contexts': [d.page_content for d in result['docs']],
        'articles': [d.metadata.get('article') for d in result['docs']],
    })

for row in capstone_baseline:
    print('Q:', row['question'])
    print('A:', row['answer'])
    print('근거:', row['articles'])
    print('-' * 80)

In [ ]:
# 다음 노트북(05_eval_ragas)과 Day2 S3 캡스톤에서 재사용하기 위해 디스크에 저장
import json
from pathlib import Path

out_path = Path('./_capstone_baseline.json')
out_path.write_text(json.dumps(capstone_baseline, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'✅ 캡스톤 베이스라인 저장: {out_path.resolve()}')
print(f'   {len(capstone_baseline)}개 질의/답변/컨텍스트 쌍')

## 정리

- LCEL은 `RunnableParallel`로 **검색 결과와 질문을 동시에 흘려보내고**, `.assign()`으로 중간 계산 산출물을 누적한다.
- **출처(citation)**을 요구하는 시스템 프롬프트는 환각 이상 탐지에 잘 들어맞는다.
- **근거 없음 → "답변할 수 없습니다"** 패턴은 생성용으로도, 이상징후 탐지용으로도 유용.

## 캡스톤 베이스라인

아래 두 산출물이 **다음 노트북(05_eval_ragas)과 Day2 Session 3 캡스톤**에서 그대로 재사용된다.

- `capstone_baseline` — 메모리 상의 리스트 (question/answer/contexts/articles)
- `./_capstone_baseline.json` — 디스크 상의 JSON (다른 노트북에서 로드 가능)

## 과제
1. `format_context`가 청크 값에 **청크 ID**를 붙이고, 프롬프트가 `[근거: 제5조#1]` 식으로 citation을 발생시키도록 바꿔보세요.
2. retriever를 `similarity_score_threshold=0.3`으로 바꾼 뒤, "내일의 날씨는?"처럼 **약관과 무관한 질의**에 시스템이 정직하게 거절하는지 확인하세요.
3. `RunnableWithMessageHistory`를 둘러 **대화형 다중턴 RAG**로 확장해보세요.

## 더 읽어보기
- teddylee777/langchain-kr — [12-RAG](https://github.com/teddylee777/langchain-kr/tree/main/12-RAG)
- teddylee777/langchain-kr — [13-LCEL](https://github.com/teddylee777/langchain-kr/tree/main/13-LCEL)
- [LangChain LCEL Interface](https://python.langchain.com/docs/concepts/lcel/)